# 08: Paired experiment reports on CPU

This notebook reads saved paired runs, their request logs, summaries and component diagnostics. It checks that checkpoint files exist but never loads model tensors. It does not train or use a GPU. Set PAIR_IDS to None for every saved pair, or list specific pair IDs.

## SETUP-01: Select saved pairs on a CPU runtime


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import pandas as pd
import torch
from google.colab import drive

if torch.cuda.is_available():
    raise RuntimeError("Switch this notebook to a CPU runtime before running it.")
drive.mount("/content/drive")
PAIR_ROOT = Path(
    "/content/drive/MyDrive/uncle/E08_forgetting_trace/paired_L9_probe/same_runtime_paired")
PAIR_IDS = None  # None reads every pair. Example: ("20261003_233622_056045",)
FOCUS_TASK = "0"
NEW_TASK = "9"

if PAIR_IDS is None:
    pair_dirs = sorted(path for path in PAIR_ROOT.iterdir() if path.is_dir())
else:
    pair_dirs = [PAIR_ROOT / pair_id for pair_id in PAIR_IDS]
if not pair_dirs:
    raise FileNotFoundError(f"No paired run folders in {PAIR_ROOT}")
for folder in pair_dirs:
    if not folder.is_dir():
        raise FileNotFoundError(folder)
print("Selected pairs:", [folder.name for folder in pair_dirs])


## REPORT-01: Read and check the saved files

Each arm has origin, environment, history, costs and summary JSON files, plus a final checkpoint. This cell reads every JSON file and checks the checkpoint's presence and size. A missing component diagnostic is shown as unavailable, not as zero change.


In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def read_arm(folder):
    history_files = list(folder.glob("history_*.json")) if folder.is_dir() else []
    if len(history_files) != 1:
        return {"status": "incomplete", "missing": [f"one history file in {folder}"]}
    stem = history_files[0].stem.removeprefix("history_")
    paths = {
        kind: folder / f"{kind}_{stem}.json"
        for kind in ("origin", "environment", "history", "costs", "summary")
    }
    checkpoint = folder / f"checkpoint_{stem}.pt"
    missing = [str(path) for path in paths.values() if not path.is_file()]
    if not checkpoint.is_file() or checkpoint.stat().st_size == 0:
        missing.append(str(checkpoint))
    if missing:
        return {"status": "incomplete", "missing": missing}

    files = {kind: read_json(path) for kind, path in paths.items()}
    history = files["history"]
    configured = files["origin"]["config"]["requests"]
    sequence_matches = (
        len(history) == len(configured)
        and all([row["action"], row["task"]] == request
                for row, request in zip(history, configured))
    )
    if not history:
        return {"status": "incomplete", "missing": ["nonempty history"]}
    last = history[-1]
    costs = files["costs"]["requests"]
    last_cost = next(
        (row for row in costs if row["index"] == last["index"]), None)
    focus_before = last["before"].get(FOCUS_TASK)
    focus_after = last["after"].get(FOCUS_TASK)
    env = files["environment"]
    runtime_keys = ("torch", "cuda", "gpu", "device", "platform", "git_commit")
    return {
        "status": "complete",
        "folder": str(folder),
        "stem": stem,
        "request_count": len(history),
        "last_request": [last["action"], last["task"]],
        "sequence_matches_config": sequence_matches,
        "focus_before": focus_before,
        "focus_after": focus_after,
        "focus_loss_points": (
            focus_before - focus_after
            if focus_before is not None and focus_after is not None else None),
        "new_task_after": last["after"].get(NEW_TASK),
        "runtime": {key: env.get(key) for key in runtime_keys},
        "new_task_sizes": env.get("task_sizes", {}).get(NEW_TASK),
        "source_sha256": files["origin"].get("starting_checkpoint_sha256"),
        "l9_seconds": last_cost["seconds"] if last_cost else None,
        "checkpoint_bytes": checkpoint.stat().st_size,
        "summary": files["summary"],
        "cost_totals": files["costs"].get("totals"),
        "file_paths": {kind: str(path) for kind, path in paths.items()},
    }

def component_rows(pair_id, component):
    if component is None:
        return []
    rows = []
    for arm_name, details in component.get("arms", {}).items():
        weights = (details.get("generated_weights")
                   or details.get("task0_generated_weights") or {})
        buffers = (details.get("task_normalization_buffers")
                   or details.get("task0_batchnorm_buffers") or {})
        for group, values in weights.items():
            rows.append({
                "pair_id": pair_id, "arm": arm_name, "component": group,
                "relative_l2_pct": values.get("relative_l2_pct"),
                "l2_change": values.get("l2_change"),
                "changed_tensors": values.get("changed_tensors"),
            })
        rows.append({
            "pair_id": pair_id, "arm": arm_name,
            "component": "BatchNorm running statistics",
            "relative_l2_pct": None,
            "l2_change": buffers.get("l2_change"),
            "changed_tensors": buffers.get("changed_tensors"),
        })
    return rows

pairs = []
arm_rows = []
all_component_rows = []
for folder in pair_dirs:
    arms = {
        name: read_arm(folder / name)
        for name in ("without_u3", "after_u3")
    }
    comparison_path = folder / "comparison.json"
    comparison = read_json(comparison_path) if comparison_path.is_file() else None
    component_path = next(
        (path for path in (folder / "component_changes.json",
                          folder / "task0_component_changes.json")
         if path.is_file()), None)
    component = read_json(component_path) if component_path else None
    complete = all(arm["status"] == "complete" for arm in arms.values())
    eligible = complete and all(
        arm["last_request"] == ["learn", NEW_TASK]
        and arm["focus_loss_points"] is not None
        and arm["sequence_matches_config"]
        for arm in arms.values())
    extra_loss = (
        arms["after_u3"]["focus_loss_points"]
        - arms["without_u3"]["focus_loss_points"]
        if eligible else None)
    stored_extra = None
    if comparison:
        stored_extra = comparison.get(
            "extra_focus_loss_after_u3_points",
            comparison.get("extra_task0_loss_after_u3_points"))
    checks = {
        "complete_arms": complete,
        "eligible_pair": eligible,
        "same_recorded_runtime": (
            arms["without_u3"]["runtime"] == arms["after_u3"]["runtime"]
            if complete else None),
    "same_new_task_sizes": (
        arms["without_u3"]["new_task_sizes"]
        == arms["after_u3"]["new_task_sizes"]
        if complete and all(arm["new_task_sizes"] is not None
                            for arm in arms.values()) else None),
        "stored_comparison_matches": (
            abs(stored_extra - extra_loss) < 1e-6
            if stored_extra is not None and extra_loss is not None else None),
        "component_diagnostic_available": component is not None,
    }
    if component is not None:
        details = component.get("arms", {})
        buffers = {
            name: (details.get(name, {}).get("task_normalization_buffers")
                   or details.get(name, {}).get("task0_batchnorm_buffers"))
            for name in ("without_u3", "after_u3")
        }
        checks["focus_buffers_unchanged"] = (
            all(value.get("changed_tensors") == 0 for value in buffers.values())
            if all(buffers.values()) else None)
        codes = {
            name: details.get(name, {}).get(
                "task_code_unchanged",
                details.get(name, {}).get("task0_code_unchanged"))
            for name in ("without_u3", "after_u3")
        }
        checks["focus_codes_unchanged"] = (
            all(value is True for value in codes.values())
            if all(value is not None for value in codes.values()) else None)
    else:
        checks["focus_buffers_unchanged"] = None
        checks["focus_codes_unchanged"] = None
    pair_record = {
        "pair_id": folder.name, "folder": str(folder),
        "arms": arms, "checks": checks,
        "extra_focus_loss_after_u3_points": extra_loss,
        "comparison_path": str(comparison_path) if comparison else None,
        "component_path": str(component_path) if component_path else None,
    }
    pairs.append(pair_record)
    for name, arm in arms.items():
        arm_rows.append({
            "pair_id": folder.name, "arm": name, "status": arm["status"],
            "focus_before": arm.get("focus_before"),
            "focus_after": arm.get("focus_after"),
            "focus_loss_points": arm.get("focus_loss_points"),
            "new_task_after": arm.get("new_task_after"),
            "l9_seconds": arm.get("l9_seconds"),
            "git_commit": arm.get("runtime", {}).get("git_commit"),
        })
    all_component_rows.extend(component_rows(folder.name, component))

display(pd.DataFrame(arm_rows))
if all_component_rows:
    display(pd.DataFrame(all_component_rows))
print("Pairs read:", len(pairs))


## REPORT-02: Build the report

Positive extra loss means task 0 fell more during L9 after U3 in that pair. This is a description of the saved runs, not proof that U3 caused the difference. The report includes missing or mismatched files so a partial run is not presented as a complete result.


In [ ]:
report = {
    "schema_version": 1,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "pair_root": str(PAIR_ROOT),
    "focus_task": FOCUS_TASK,
    "new_task": NEW_TASK,
    "pairs": pairs,
    "component_rows": all_component_rows,
    "scope": "Saved-run comparison only. A pair does not establish a general U3 effect.",
}

def shown(value):
    return "n/a" if value is None else f"{value:.2f}"

lines = [
    "# Paired L9 run report", "",
    f"Generated: {report['generated_at_utc']}", "",
    "Positive extra loss means task 0 fell more during L9 after U3.",
    "Each pair is one observation. These numbers do not establish a general effect.",
    "",
    "| Pair | Without U3 loss (pp) | After U3 loss (pp) | Extra loss (pp) | Same runtime | Components |",
    "| --- | ---: | ---: | ---: | --- | --- |",
]
for item in pairs:
    arms = item["arms"]
    checks = item["checks"]
    lines.append(
        f"| {item['pair_id']} | "
        f"{shown(arms['without_u3'].get('focus_loss_points'))} | "
        f"{shown(arms['after_u3'].get('focus_loss_points'))} | "
        f"{shown(item['extra_focus_loss_after_u3_points'])} | "
        f"{checks['same_recorded_runtime']} | "
        f"{'available' if checks['component_diagnostic_available'] else 'unavailable'} |")

for item in pairs:
    lines += ["", f"## {item['pair_id']}", ""]
    for name, arm in item["arms"].items():
        lines.append(
            f"- {name}: task {FOCUS_TASK} "
            f"{shown(arm.get('focus_before'))}% to "
            f"{shown(arm.get('focus_after'))}%; "
            f"L9 loss {shown(arm.get('focus_loss_points'))} points.")
        if arm["status"] != "complete":
            lines.append(f"  Missing: {', '.join(arm.get('missing', []))}")
    lines.append(f"- Checks: {item['checks']}")
    lines.append(f"- Component diagnostic: "
                 f"{item['component_path'] or 'unavailable'}")
    details = [row for row in all_component_rows
               if row["pair_id"] == item["pair_id"]]
    if details:
        lines += [
            "",
            "| Arm | Component | Relative weight change (%) | Changed tensors |",
            "| --- | --- | ---: | ---: |",
        ]
        for row in details:
            lines.append(
                f"| {row['arm']} | {row['component']} | "
                f"{shown(row['relative_l2_pct'])} | "
                f"{row['changed_tensors'] if row['changed_tensors'] is not None else 'n/a'} |")

report_dir = PAIR_ROOT.parent / "analysis"
report_dir.mkdir(parents=True, exist_ok=True)
json_path = report_dir / "paired_L9_report.json"
markdown_path = report_dir / "paired_L9_report.md"
for path, content in (
    (json_path, json.dumps(report, indent=2) + "\n"),
    (markdown_path, "\n".join(lines) + "\n"),
):
    temporary = path.with_suffix(path.suffix + ".writing")
    temporary.write_text(content, encoding="utf-8")
    os.replace(temporary, path)
assert read_json(json_path) == report
print("Saved:", json_path)
print("Saved:", markdown_path)
display(pd.DataFrame([{
    "pair_id": item["pair_id"],
    "without_u3_loss": item["arms"]["without_u3"].get("focus_loss_points"),
    "after_u3_loss": item["arms"]["after_u3"].get("focus_loss_points"),
    "extra_loss": item["extra_focus_loss_after_u3_points"],
    "checks": item["checks"],
} for item in pairs]))


## FINISH-01: Flush Drive

The report is saved before Drive is unmounted. This notebook runs on CPU, so there is no GPU to release.


In [ ]:
assert json_path.is_file() and markdown_path.is_file()
drive.flush_and_unmount(timeout_ms=600000)
print("Drive flushed. CPU report complete.")
